# Paper Digest — Colab ML for Keywording and Query Intent

選定コーパスの探索的ベースラインです。R2V カタログの件数にかかわらず、**実在する PDF テキストとレビュー済みラベルが揃うまでスコアを報告しない**こと。旧 Kaggle 版: `notebooks/kaggle/keyword-intent-ml.ipynb`。

目標: (1) チャンクに紐づく候補キーワード、(2) 人手レビュー済みラベルによる研究質問インテント分類、(3) 出所情報付きの結果エクスポート。

In [ ]:
# Install the ML dependencies
%pip -q install pandas numpy scikit-learn matplotlib

## 0. Repository setup

In [ ]:
# Clone the canonical repository (re-run safe)
from pathlib import Path
import subprocess

REPO_DIR = Path('/content/paper-digest-src')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/bonsai/paper-digest.git', str(REPO_DIR)], check=True)
print('Repo ready at', REPO_DIR)

## 1. Load the real chunk corpus

`chunks.jsonl` は PDF ingestion パイプライン（`paper_digest_colab.ipynb`）の生成物で、`outputs/latest/chunks.jsonl` にあります。必須フィールド: `paper_id`, `chunk_id`, `page_start`, `text`。プレースホルダの研究テキストは生成しません。

In [ ]:
import pandas as pd

CHUNKS_PATH = REPO_DIR / 'outputs' / 'latest' / 'chunks.jsonl'
OUT_DIR = REPO_DIR / 'outputs' / 'ml'
OUT_DIR.mkdir(parents=True, exist_ok=True)

if CHUNKS_PATH.exists():
    chunks = pd.read_json(CHUNKS_PATH, lines=True)
    required = {'paper_id', 'chunk_id', 'page_start', 'text'}
    missing = required - set(chunks.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    chunks = chunks.dropna(subset=['text']).copy()
    chunks['text'] = chunks['text'].astype(str)
    print('Chunks:', len(chunks), 'Papers:', chunks['paper_id'].nunique())
else:
    chunks = pd.DataFrame(columns=['paper_id', 'chunk_id', 'page_start', 'text'])
    print('outputs/latest/chunks.jsonl not found. Run paper_digest_colab.ipynb first.')

## 2. Keywording baseline: TF-IDF

チャンク単位で語を抽出し、出所識別子を保持します。これは語彙ベースのベースラインであり、意味理解モデルではありません。候補語は人手でレビューしてからグラフへ追加してください。

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

def extract_keyword_candidates(frame: pd.DataFrame, top_k: int = 12) -> pd.DataFrame:
    columns = ['paper_id', 'chunk_id', 'page_start', 'keyword', 'score', 'method', 'review_status']
    if frame.empty:
        return pd.DataFrame(columns=columns)
    vectorizer = TfidfVectorizer(
        stop_words='english',
        ngram_range=(1, 2),
        max_df=0.95,
        min_df=1,
        token_pattern=r'(?u)[a-zA-Z][a-zA-Z0-9_-]{2,}',
    )
    matrix = vectorizer.fit_transform(frame['text'].tolist())
    terms = np.asarray(vectorizer.get_feature_names_out())
    rows = []
    for row_idx, row in enumerate(frame.itertuples(index=False)):
        scores = matrix[row_idx].toarray().ravel()
        if not scores.any():
            continue
        for term_idx in np.argsort(scores)[-top_k:][::-1]:
            if scores[term_idx] <= 0:
                continue
            rows.append({
                'paper_id': row.paper_id,
                'chunk_id': row.chunk_id,
                'page_start': row.page_start,
                'keyword': terms[term_idx],
                'score': float(scores[term_idx]),
                'method': 'tfidf_chunk_baseline',
                'review_status': 'unreviewed',
            })
    return pd.DataFrame(rows)

keyword_candidates = extract_keyword_candidates(chunks)
keyword_candidates.to_csv(OUT_DIR / 'keyword_candidates.csv', index=False)
keyword_candidates.head(30)

## 3. Query-intent dataset

人手レビュー済み CSV（`query_text` と `intent_label`）を repo の `inputs/labels/intent_examples.csv` に置いてください。任意フィールド: `query_id`, `split_group`, `review_status`。合成テスト例は明示的にラベルし、実ユーザ行動と混同しないこと。

初期ラベル: `summarize`, `compare`, `find_method`, `find_dataset`, `find_metric`, `find_limitations`, `find_evidence`, `identify_gap`, `other_or_unclear`。

In [ ]:
INTENT_PATH = REPO_DIR / 'inputs' / 'labels' / 'intent_examples.csv'
if INTENT_PATH.exists():
    intents = pd.read_csv(INTENT_PATH)
    required = {'query_text', 'intent_label'}
    missing = required - set(intents.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')
    if 'review_status' in intents.columns:
        intents = intents[intents['review_status'].eq('reviewed')].copy()
    print('Reviewed examples:', len(intents), 'Classes:', intents['intent_label'].nunique())
else:
    intents = pd.DataFrame(columns=['query_text', 'intent_label'])
    print('No reviewed intent labels found. Classifier training is skipped.')

## 4. Train a small baseline only when data supports it

10 本の論文コーパスはラベル付きクエリデータセットの代替にはなりません。このセルは 1 クラスあたり 2 例以上かつ層化ホールドアウトに十分なサンプルがある場合のみ学習し、それ以外は意図的に訓練をスキップします。小さなデータセットではルールベースのルーティングを使い、まずラベルを集めること。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

intent_model = None
if len(intents) >= 20 and intents['intent_label'].value_counts().min() >= 2:
    X_train, X_test, y_train, y_test = train_test_split(
        intents['query_text'], intents['intent_label'],
        test_size=0.25, random_state=42, stratify=intents['intent_label'],
    )
    intent_model = Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), lowercase=True, sublinear_tf=True)),
        ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced')),
    ])
    intent_model.fit(X_train, y_train)
    predictions = intent_model.predict(X_test)
    print(classification_report(y_test, predictions, zero_division=0))
else:
    print('Skipped: collect more human-reviewed intent labels before training or reporting metrics.')

if intent_model is not None:
    pd.DataFrame({
        'query_text': X_test,
        'actual_intent': y_test,
        'predicted_intent': predictions,
    }).to_csv(OUT_DIR / 'intent_predictions.csv', index=False)

## 5. Review and export

未レビューのキーワードを正準グラフへ直接流し込まないこと。手動サンプルで precision@k を確認し、インテント混同を監査し、受理したキーワードには元チャンク/ページを保持します。

出力は `outputs/ml/` に保存されます。これは公開契約（`outputs/latest/`）の一部ではなく、人手レビュー後に commit してください。このノートブックはまだ実ユーザ PDF に対して実行されておらず、実測スコアは存在しません。